Facebook data downloaded from Humanitarian Data Exchange on 17th January 2025.
Links for 2020 and 2021-2022 data are: https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/55a51014-0d27-49ae-bf92-c82a570c2c6c/download/movement-range-data-2022-05-22.zip
and https://data.humdata.org/dataset/c3429f0e-651b-4788-bb2f-4adbf222c90e/resource/3d77ce5c-ab6d-4864-b8a2-c8bafffac4f3/download/movement-range-data-2020-03-01-2020-12-31.zip
which are accessible from:
https://data.humdata.org/dataset/movement-range-maps?

In [ ]:
import rioxarray as rx
import geopandas as gp
import pyproj
import pandas as pd
import polars as pl
import numpy as np
import pycountry
import json
import urllib.request

from emu_renewal.geospatial import DATA_PATH, RAW_MOB_PATH, raster_to_polydf

# Stop GeoPandas warning about area intersection calculations
import warnings
warnings.filterwarnings("ignore", "Geometry is in a geographic CRS")

In [ ]:
country = "Denmark"
iso3 = pycountry.countries.lookup(country).alpha_3

# Usually level 1 for European countries and 2 for other, but check in Facebook movement table
gadm_level = 1

In [ ]:
# Gridded Population of the World 30sec 2020 dataset
# https://www.earthdata.nasa.gov/data/projects/gpw
pop_ds = rx.open_rasterio(DATA_PATH / "population/gpw_v4_population_count_rev11_2020_30_sec.tif")

# Alternative dataset from:
# https://github.com/lulingliu/GlobPOP
#pop_ds = rx.open_rasterio(DATA_PATH / "population/GlobPOP_Count_30arc_2020_I32.tiff")

In [ ]:
# Download the appropriate GADM boundaries json
source = f"https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_{iso3}_{gadm_level}.json.zip"
dest = DATA_PATH / f"population/gadm_input_json/gadm41_{iso3}_{gadm_level}.json.zip"
if not dest.exists():
    urllib.request.urlretrieve(source, dest)

# GADM administrative boundaries as obtained from:
# https://gadm.org/download_country.html

# These files are directly downloadable as of 21/01/2025 via
# https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_{iso3_code}_{gadm_level}.json.zip

poly_df = gp.read_file(dest)

In [ ]:
# Set up the dimensions of a population cell
pix_dim = float((pop_ds.coords["x"][1] - pop_ds.coords["x"][0]).data)

pop_dict = {}

# Loop over the polygons relevant to the country
for i_poly, poly_id in enumerate(poly_df[f"GID_{gadm_level}"]):

    # Get the relevant gridded population data, ensuring correct projection
    poly_bounds = np.array(poly_df.loc[i_poly, "geometry"].bounds)  # Furthest reach of polygon in each direction
    expanded_bounds = poly_bounds + np.array(([-pix_dim, -pix_dim, pix_dim, pix_dim]))  # Extend bounds to span every population data coordinate that could be relevant
    pop_clipped = pop_ds.rio.clip_box(*expanded_bounds)
    pop_df = raster_to_polydf(pop_clipped, "population")
    pop_df = pop_df.set_crs(pyproj.CRS.from_wkt(pop_ds.rio.crs.to_wkt()))  # Reconcile projection of polygon and population data

    # Find population based on intersections
    isect = gp.overlay(pop_df, poly_df.iloc[i_poly: i_poly + 1], keep_geom_type=False)
    pop_val = float((isect.area / pix_dim ** 2.0 * isect.population).sum())
    pop_dict[poly_id] = pop_val
    print(f"{poly_id} has population of {round(pop_val / 1e3)} thousand")

In [ ]:
json.dump(pop_dict, open(DATA_PATH / f"population/gadm_est/{iso3}_{gadm_level}.json", "w"))

In [ ]:
f"Total population for country is {round(sum(pop_dict.values()) / 1e6, 3)} million"

In [ ]:
data_col = "all_day_bing_tiles_visited_relative_change"
iso3 = pycountry.countries.lookup(country).alpha_3

# Should be 1 for most European countries, 2 for most others
gadm_level = 1

In [ ]:
# Compile Facebook data
data20 = pl.read_csv(RAW_MOB_PATH / "movement-range-data-2020-03-01--2020-12-31.txt", separator="\t", schema_overrides={"ds": pl.datatypes.Date})
data21_22 = pl.read_csv(RAW_MOB_PATH / "movement-range-2022-05-22.txt", separator="\t", schema_overrides={"ds": pl.datatypes.Date})
fb_data = pl.concat([data20, data21_22])
country_mobility = fb_data.filter(pl.col("country") == iso3)

# Get country population data produced by gridded population notebook
country_pop_data = json.load(open(DATA_PATH/f"population/gadm_est/{iso3}_{gadm_level}.json", "r"))

In [ ]:
# Calculate weighted average over patches
country_mob_series = pd.Series(0.0, index=country_mobility["ds"].unique(), dtype=float)
total_pop = 0.0
for pid in country_pop_data:
    if pid in country_mobility["polygon_id"]:
        cur_data = country_mobility.filter(pl.col("polygon_id") == pid)
        mob_series = pd.Series(index=cur_data["ds"].unique(), data=cur_data[data_col]).dropna()
        region_pop = country_pop_data[pid]
        country_mob_series += mob_series.reindex(country_mob_series.index, method="nearest") * region_pop
        total_pop += region_pop
weighted_country_mob = country_mob_series / total_pop

In [ ]:
weighted_country_mob.to_csv(DATA_PATH / f"mobility/{iso3}_fbmob_data.csv")